# 1. Imports

In [1]:
import os
import weave
import wandb
import torch
import pandas as pd
import os.path as osp
import io
import base64
import re
import trl
import inspect
import time
from openai import OpenAI
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration
from models.mllm_wrappers import LLaVA, Qwen, Janus, Qwen3_5, Molmo
from peft import LoraConfig, get_peft_model
from configuration_template import Config
from bert_score import BERTScorer
from data_prep.data_utils import build_grid_image, build_patch_image_with_target_highlight
from trl import GRPOConfig, GRPOTrainer
from sentence_transformers import SentenceTransformer, util




/home/alesau/miniconda3/envs/mllm/lib/python3.14/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


In [2]:
os.environ["WANDB_API_KEY"] = "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"
wandb.login(key="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")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /home/alesau/.netrc
wandb: Currently logged in as: alex-esau98 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
#!pip install -U transformers accelerate datasets peft trl qwen-vl-utils wandb weave bitsandbytes
#!pip install wandb
#!pip install weave

: 

: 

# 2. Inits

### Patches

In [2]:


weave.init('alex-esau98/mllm')


wandb.init(
    entity="alex-esau98",
    project="mllm",
    config={
        "model": "Qwen/Qwen2-VL-2B-Instruct",
        "method": "GRPO",
        "reward": "judge_accuracy_plus_penalties"
    }
)

weave: weave version 0.53.6 is available!  To upgrade, please run:
weave:  $ pip install weave --upgrade
weave: Logged in as Weights & Biases user: alex-esau98.
weave: View Weave data at https://wandb.ai/alex-esau98/mllm/weave
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/alesau/.netrc.
wandb: Currently logged in as: alex-esau98 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Initializing weave.


In [3]:
config = Config(model="Qwen/Qwen2-VL-2B-Instruct")
config.input_type = 'simple'
config.task = 'grid'

img_kwargs = {
        "patch_size": config.patch_size,
        "patch_padding": config.patch_padding,
        "patch_pad_color": config.patch_pad_color,
    }

generate_kwargs = {
        "max_new_tokens": config.max_new_tokens,
        "do_sample": True,
        "temperature": 0.8
    }

model_kwargs = {"cache_dir": config.model_cache_dir, "quant": config.quant}

Model = Qwen
model = Model(config.model_id, **model_kwargs)
print(f'Model: {model.__class__}')

model_version: Qwen2
model_type: Qwen
/home/alesau/mllm/mllm/PATH
building Qwen model...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Model: <class 'models.mllm_wrappers.Qwen'>


In [4]:
config_judge = Config(model="Qwen/Qwen3.5-0.8B")
config_judge.input_type = 'simple'
config_judge.task = 'grid'

img_kwargs_judge = {
        "patch_size": config.patch_size,
        "patch_padding": config.patch_padding,
        "patch_pad_color": config.patch_pad_color,
    }

generate_kwargs_judge = {
        "max_new_tokens": config_judge.max_new_tokens,
        "do_sample": True,
        "temperature": 0.8
    }       

model_kwargs_judge = {"cache_dir": config_judge.model_cache_dir, "quant": config_judge.quant}

JudgeModel = Qwen3_5
judge_model = JudgeModel(config_judge.model_id, **model_kwargs_judge)
print(f'Judge Model: {judge_model.__class__}')

#judge_model = OpenAI(
#    api_key="EMPTY",
#    base_url="http://129.70.40.28:8000/v1",
#)


model_version: Qwen3.5
model_type: Qwen3_5
/home/alesau/mllm/mllm/PATH
building Qwen3_5 model...


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

Judge Model: <class 'models.mllm_wrappers.Qwen3_5'>


In [5]:

print(type(model))
print(model.__dict__.keys())
print(type(model.model))
print(type(model.processor))

print()
print(type(judge_model))
print(judge_model.__dict__.keys())
print(type(judge_model.model))
print(type(judge_model.processor))
print(judge_model.model_size)

<class 'models.mllm_wrappers.Qwen'>
dict_keys(['model', 'processor', 'model_id', 'model_size', 'quant', 'device'])
<class 'transformers.models.qwen2_vl.modeling_qwen2_vl.Qwen2VLForConditionalGeneration'>
<class 'transformers.models.qwen2_vl.processing_qwen2_vl.Qwen2VLProcessor'>

<class 'models.mllm_wrappers.Qwen3_5'>
dict_keys(['model', 'processor', 'model_id', 'model_size', 'quant', 'device'])
<class 'transformers.models.qwen3_5.modeling_qwen3_5.Qwen3_5ForConditionalGeneration'>
<class 'transformers.models.qwen3_vl.processing_qwen3_vl.Qwen3VLProcessor'>
0.8b


In [6]:


hf_model = model.model
processor = model.processor

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],

    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)

hf_model = get_peft_model(hf_model, lora_config)

hf_model.print_trainable_parameters()

trainable params: 4,358,144 || all params: 2,213,343,744 || trainable%: 0.1969


### Grids

In [7]:


weave.init('alex-esau98/mllm')


wandb.init(
    entity="alex-esau98",
    project="mllm",
    config={
        "model": "Qwen/Qwen2-VL-2B-Instruct",
        "method": "GRPO",
        "reward": "judge_accuracy_plus_penalties"
    }
)

Output()

weave: weave version 0.53.6 is available!  To upgrade, please run:
weave:  $ pip install weave --upgrade
weave: Logged in as Weights & Biases user: alex-esau98.
weave: View Weave data at https://wandb.ai/alex-esau98/mllm/weave


wandb: Initializing weave.


In [ ]:
config = Config(model="Qwen/Qwen2-VL-2B-Instruct")
config.input_type = 'simple'
config.task = 'grid'

img_kwargs = {
        "patch_size": config.patch_size,
        "patch_padding": config.patch_padding,
        "grid_padding": config.grid_padding,
        "patch_pad_color": config.patch_pad_color,
        "grid_pad_color": config.grid_pad_color,
    }

generate_kwargs = {
        "max_new_tokens": config.max_new_tokens,
        "do_sample": True,
        "temperature": 0.8
    }

model_kwargs = {"cache_dir": config.model_cache_dir, "quant": config.quant}

Model = Qwen
model = Model(config.model_id, **model_kwargs)
print(f'Model: {model.__class__}')

model_version: Qwen2
model_type: Qwen
/home/alesau/mllm/mllm/PATH
building Qwen model...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Model: <class 'models.mllm_wrappers.Qwen'>


In [ ]:
config_judge = Config(model="Qwen/Qwen3.5-9B")
config_judge.input_type = 'simple'
config_judge.task = 'grid'

img_kwargs_judge = {
        "patch_size": config.patch_size,
        "patch_padding": config.patch_padding,
        "grid_padding": config.grid_padding,
        "patch_pad_color": config.patch_pad_color,
        "grid_pad_color": config.grid_pad_color,
    }

generate_kwargs_judge = {
        "max_new_tokens": config_judge.max_new_tokens,
        "do_sample": True,
        "temperature": 0.8
    }       

model_kwargs_judge = {"cache_dir": config_judge.model_cache_dir, "quant": config_judge.quant}

#JudgeModel = Qwen3_5
#judge_model = JudgeModel(config_judge.model_id, **model_kwargs_judge)
#print(f'Judge Model: {judge_model.__class__}')

from openai import OpenAI

judge_model = OpenAI(
    api_key="EMPTY",
    base_url="http://129.70.40.28:8000/v1",
)


model_version: Qwen3.5
model_type: Qwen3_5
/home/alesau/mllm/mllm/PATH


In [7]:

print(type(model))
print(model.__dict__.keys())
print(type(model.model))
print(type(model.processor))

print()
print(type(judge_model))
print(judge_model.__dict__.keys())
print(type(judge_model.model))
print(type(judge_model.processor))
print(judge_model.model_size)

<class 'models.mllm_wrappers.Qwen'>
dict_keys(['model', 'processor', 'model_id', 'model_size', 'quant', 'device'])
<class 'transformers.models.qwen2_vl.modeling_qwen2_vl.Qwen2VLForConditionalGeneration'>
<class 'transformers.models.qwen2_vl.processing_qwen2_vl.Qwen2VLProcessor'>

<class 'models.mllm_wrappers.Qwen3_5'>
dict_keys(['model', 'processor', 'model_id', 'model_size', 'quant', 'device'])
<class 'transformers.models.qwen3_5.modeling_qwen3_5.Qwen3_5ForConditionalGeneration'>
<class 'transformers.models.qwen3_vl.processing_qwen3_vl.Qwen3VLProcessor'>
9b


In [ ]:


hf_model = model.model
processor = model.processor

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],

    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)

hf_model = get_peft_model(hf_model, lora_config)

hf_model.print_trainable_parameters()

trainable params: 4,358,144 || all params: 2,213,343,744 || trainable%: 0.1969


# 3. Data Prep

### Patches

In [8]:
data_df = pd.read_json(osp.join(config.data_dir, "color_patch_data_far.json"))
print(len(data_df))
dataset_train = data_df.sample(frac=0.12665, random_state=42) ##grid = 0.55  patches = frac=0.12665
dataset_judge = data_df.drop(dataset_train.index)

print(len(dataset_train))
print(len(dataset_judge))
data_df

#for i in range(70,80):
#    print(data_df["conversation"][i])

15791
2000
13791


,identifier,gameid,roundNum,condition,success,hls_values,s_order_t_distr1_distr2,l_order_t_distr1_distr2,conversation,n_utterances
0,1124-1:2,1124-1,2,far,True,"[[249, 50, 25], [129, 50, 66], [54, 50, 53]]","[0, 1, 2]","[1, 0, 2]","[[speaker, purple, None]]",1
1,1124-1:5,1124-1,5,far,True,"[[156, 50, 84], [57, 50, 64], [239, 50, 87]]","[0, 2, 1]","[2, 0, 1]","[[speaker, Mint green., None]]",1
2,1124-1:6,1124-1,6,far,True,"[[40, 50, 7], [288, 50, 10], [58, 50, 62]]","[0, 1, 2]","[0, 1, 2]","[[speaker, Mud brown, None]]",1
3,1124-1:10,1124-1,10,far,True,"[[215, 50, 13], [164, 50, 81], [92, 50, 84]]","[2, 1, 0]","[2, 1, 0]","[[speaker, Grey, None]]",1
4,1124-1:11,1124-1,11,far,True,"[[229, 50, 31], [172, 50, 65], [347, 50, 52]]","[0, 1, 2]","[2, 1, 0]","[[speaker, Purple, None]]",1
...,...,...,...,...,...,...,...,...,...,...
15786,8452-5:42,8452-5,42,far,True,"[[261, 50, 24], [57, 50, 79], [61, 50, 12]]","[0, 2, 1]","[1, 0, 2]","[[speaker, purple, None]]",1
15787,8452-5:43,8452-5,43,far,True,"[[144, 50, 31], [226, 50, 94], [219, 50, 7]]","[1, 2, 0]","[0, 2, 1]","[[speaker, green, None]]",1
15788,8452-5:44,8452-5,44,far,True,"[[189, 50, 73], [81, 50, 40], [29, 50, 2]]","[1, 0, 2]","[0, 2, 1]","[[speaker, blue, None]]",1
15789,8452-5:45,8452-5,45,far,True,"[[352, 50, 20], [139, 50, 91], [58, 50, 44]]","[1, 0, 2]","[2, 1, 0]","[[speaker, pink, None]]",1


### Grids

In [ ]:

data_df = pd.read_json(osp.join(config.data_dir, "color_grid_data_far.json"))
print(len(data_df))
dataset_train = data_df.sample(frac=0.5502, random_state=42) ##grid = 0.55  patches = frac=0.12665
dataset_judge = data_df.drop(dataset_train.index)

print(len(dataset_train))
print(len(dataset_judge))
data_df

#for i in range(70,80):
#    print(data_df["conversation"][i])

3635
2000
1635


,gameid,game_idx,roundNum,round_id,condition,success,utterances,n_utterances,objs,target,speaker_order,listener_order,listener_clicked
0,4147-9d9582f2-28d0-431f-b6ba-35fd1be76619,1,1,4147-9d9582f2-28d0-431f-b6ba-35fd1be76619_1,FAR,True,"[[speaker, green top left corner, 1528144087280]]",1,"[{'shapes': [{'color': [322, 21, 50]}, {'color...",1,"[2, 1, 0]","[1, 0, 2]",0
1,4147-9d9582f2-28d0-431f-b6ba-35fd1be76619,1,4,4147-9d9582f2-28d0-431f-b6ba-35fd1be76619_4,FAR,True,"[[speaker, green in the middle square top row,...",1,"[{'shapes': [{'color': [294, 7, 50]}, {'color'...",0,"[1, 2, 0]","[2, 1, 0]",2
2,4147-9d9582f2-28d0-431f-b6ba-35fd1be76619,1,11,4147-9d9582f2-28d0-431f-b6ba-35fd1be76619_11,FAR,True,"[[speaker, bottom left not yellow not orange, ...",1,"[{'shapes': [{'color': [339, 87, 50]}, {'color...",1,"[1, 2, 0]","[0, 2, 1]",2
3,4147-9d9582f2-28d0-431f-b6ba-35fd1be76619,1,12,4147-9d9582f2-28d0-431f-b6ba-35fd1be76619_12,FAR,True,"[[speaker, top middle is grey, 1528144497114]]",1,"[{'shapes': [{'color': [199, 26, 50]}, {'color...",1,"[0, 1, 2]","[1, 0, 2]",0
4,4147-9d9582f2-28d0-431f-b6ba-35fd1be76619,1,15,4147-9d9582f2-28d0-431f-b6ba-35fd1be76619_15,FAR,True,"[[speaker, top right is green, 1528144581184]]",1,"[{'shapes': [{'color': [329, 14, 50]}, {'color...",2,"[2, 1, 0]","[2, 1, 0]",0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3630,6068-c8135454-2875-41d9-918c-60c34fc9460c,198,45,6068-c8135454-2875-41d9-918c-60c34fc9460c_45,FAR,True,"[[speaker, Center of middle row is green, cent...",1,"[{'shapes': [{'color': [345, 11, 50]}, {'color...",1,"[2, 0, 1]","[2, 0, 1]",2
3631,6068-c8135454-2875-41d9-918c-60c34fc9460c,198,47,6068-c8135454-2875-41d9-918c-60c34fc9460c_47,FAR,True,"[[speaker, middle tile and middle bottom are b...",1,"[{'shapes': [{'color': [354, 73, 50]}, {'color...",1,"[0, 2, 1]","[0, 1, 2]",1
3632,6068-c8135454-2875-41d9-918c-60c34fc9460c,198,52,6068-c8135454-2875-41d9-918c-60c34fc9460c_52,FAR,True,"[[speaker, top left green, 1526435104682]]",1,"[{'shapes': [{'color': [277, 19, 50]}, {'color...",1,"[2, 0, 1]","[1, 2, 0]",0
3633,6068-c8135454-2875-41d9-918c-60c34fc9460c,198,57,6068-c8135454-2875-41d9-918c-60c34fc9460c_57,FAR,True,"[[speaker, Top middle green, 1526435245005]]",1,"[{'shapes': [{'color': [195, 34, 50]}, {'color...",1,"[2, 1, 0]","[1, 2, 0]",0


# 4. Methods and Dataset

### Patches

In [11]:
####patches

scorer = BERTScorer(lang="en", rescale_with_baseline=True)

wandb.define_metric("reward/local_step")
wandb.define_metric("reward/*", step_metric="reward/local_step")

def build_prompt(target_position):
    return f"""
Describe the color patch {target_position} so that another model can identify it among three patches.

Rules:

- Do not use left, middle, right, center.
- Describe only visual properties.
- Mention hue, saturation, brightness, contrast, texture if useful.
- Keep it under 30 words.
"""



def parse_position(text):
    text = text.lower().strip()

    if "left" in text:
        return "left"

    if "middle" in text or "center" in text:
        return "middle"

    if "right" in text:
        return "right"

    return "invalid"



@weave.op()
def judge_patch_from_description(image, description, judge_order):

    judge_prompt = f"""
You see an image with three color patches: left, middle, right.

A description says:
"{description}"

Which patch is described?

Answer only:
left, middle, or right.
"""

    # Hier dein Judge-VLM aufrufen
    img = build_patch_image_with_target_highlight(image, judge_order, **img_kwargs, highlight_target=False)
    #display(img)
    #print(f"Generated Description: {description}")
    answer = run_vlm_judge(img, judge_prompt)
    #print(f"Judge Answer: {answer}")
    #print("-" * 50)

    return  {
        "position": parse_position(answer),
        "image": img
    }


def run_vlm_judge(image, prompt):
    # Hier rufst du dein Judge-VLM mit dem Bild und Prompt auf und bekommst die Antwort zurück
    # Zum Beispiel:
    # answer = judge_vlm.generate(image=image, prompt=prompt, **generate_kwargs)
    # return answer
    response = judge_model.generate(prompt, image, **generate_kwargs_judge)
    
    # buffer = io.BytesIO()
    # image.save(buffer, format="PNG")
    # img_base64 = base64.b64encode(buffer.getvalue()).decode("utf-8")

    # image_url = f"data:image/png;base64,{img_base64}"

    # response = judge_model.chat.completions.create(
    #     model="Qwen/Qwen3.6-27B-FP8",
    #     messages=[
    #         {
    #             "role": "user",
    #             "content": [
    #                 {
    #                     "type": "image_url",
    #                     "image_url": {
    #                         "url": image_url
    #                     }
    #                 },
    #                 {
    #                     "type": "text",
    #                     "text": prompt
    #                 },
    #             ],
    #         }
    #     ],
    #     temperature=0.7,
    #     #max_tokens=800,
    #     extra_body={
    #         "chat_template_kwargs": {
    #             "enable_thinking": False
    #         },
    #         "thinking_token_budget": 40
    #     }
    # )

    # response = response.choices[0].message.content


    
    
    
    
    
    # Dummy-Implementierung für das Beispiel
    return response





def contains_position_words(text):

    forbidden = [
        "left",
        "middle",
        "right",
        "center",
        "centre",
    ]

    return any(w in text.lower() for w in forbidden)






@weave.op()
def compute_reward(description, image, target_position, judge_order):

    reward = 0.0

    judge_prediction = judge_patch_from_description(
        image=image,
        description=description,
        judge_order=judge_order
    )


    # Judge richtig?
    if judge_prediction == target_position:
        reward += 1.0
    else:
        reward -= 0.2

    # Cheating bestrafen
    #if contains_position_words(description):
    #    reward -= 0.5

    # Zu kurz
    word_count = len(description.split())

    # Zu lang
    #reward -= word_count * 0.015

    return {
        "reward": reward,
        "judge_prediction": judge_prediction,
        "word_count": word_count,
        "contains_position_words":
            contains_position_words(description)
    }






@weave.op()
def compute_reward_bert(description, image, target_position, judge_order, og_description):
    global scorer

    reward = 0.0

    F1, _, _ = scorer.score([description], [og_description])

    bert_weight = 0.5  # Gewichtung für den BERT-Score
    judge_weight = 1-bert_weight

    bert_reward = bert_weight * F1[0].item()
    reward += bert_reward  # F1-Score als Belohnung hinzufügen

    result = judge_patch_from_description(
        image=image,
        description=description,
        judge_order=judge_order
    )

    judge_prediction = result["position"]
    img = result["image"]

    # Judge richtig?
    prediction_true = judge_prediction == target_position

    if prediction_true:
        judge_reward = 1.0* judge_weight
    else:
        judge_reward = -0.2* judge_weight
    reward += judge_reward

    # Cheating bestrafen
    #if contains_position_words(description):
    #    reward -= 0.5

    # Zu kurz
    word_count = len(description.split())

    # Zu lang
    #reward -= word_count * 0.015

    return {
        "reward": reward,
        "judge_prediction": judge_prediction,
        "word_count": word_count,
        "contains_position_words":
            contains_position_words(description),
        "image": img,
        "bert_reward": bert_reward,
        "judge_reward": judge_reward,
        "prediction_true": prediction_true
    }




all_rewards = []
sample_rewards_avg = []
judge_correct_avg = []
word_count_avg = []
contains_position_words_avg = []
reward_local_step = 0




def reward_func(completions, patches, rl_order, judge_order, og_description, **kwargs):
    global all_rewards
    global reward_local_step

    rewards = []
    #print(type(completions[0]))
    #print(completions[0])
    t_reward_start = time.perf_counter()
    for desc, img, rl_ord, judge_ord, og_desc in zip(
        completions,
        patches,
        rl_order,
        judge_order,
        og_description
    ):
        desc = desc[0]["content"]
        
        target_position = None
        if judge_ord[0] == 0:
            target_position = "left"
        elif judge_ord[1] == 0:
            target_position = "middle"
        elif judge_ord[2] == 0:
            target_position = "right"



        """result = compute_reward(
            description=desc,
            image=img,
            target_position=target_position,
            judge_order=judge_ord
        )"""

        result = compute_reward_bert(
            description=desc,
            image=img,
            target_position=target_position,
            judge_order=judge_ord,
            og_description=og_desc
        )

        

        rewards.append(result["reward"])

        reward_local_step += 1

        wandb.log({
            "reward/local_step": 
                reward_local_step,
                
            "reward/sample_reward":
                result["reward"],

            "reward/judge_correct":
                int(result["judge_prediction"] == target_position),

            "reward/word_count":
                result["word_count"],

            "reward/contains_position_words":
                int(result["contains_position_words"])
        })

    t_reward_end = time.perf_counter()
    print(f"Reward total: {t_reward_end - t_reward_start:.2f}s")

    all_rewards.append(rewards)
    return rewards


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [12]:
##patches

train_dataset = []
for i, entry in dataset_train.iterrows():
    #print(f"Entry {i}: {entry}")


    if entry.s_order_t_distr1_distr2[0] == 0:
        target_position = "on the left"
    elif entry.s_order_t_distr1_distr2[1] == 0:
        target_position = "in the middle"
    elif entry.s_order_t_distr1_distr2[2] == 0:
        target_position = "on the right"
    img = build_patch_image_with_target_highlight(entry.hls_values, entry.s_order_t_distr1_distr2, **img_kwargs, highlight_target=False)
    prompt = build_prompt(target_position)

    utterances = entry.conversation
    conversation = ""
    for utt in utterances:
        if conversation == "":
            conversation += f"{utt[1]}"
        else:
            conversation += f", {utt[1]}"
    conversation = re.sub(r"[^A-Za-zÄÖÜäöüß,\s]", "", conversation)
    

    
    #print(f"Prompt: {prompt}")
    #display(img)
    sample = {
        "prompt": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "image", 
                        "image": img,
                    },
                    {
                        "type": "text",
                        "text": prompt
                    }
                ]
            }
        ],

        "patches": entry.hls_values,
        "rl_order": entry.s_order_t_distr1_distr2,
        "judge_order": entry.l_order_t_distr1_distr2,
        "og_description": conversation
    }
    train_dataset.append(sample)

train_dataset

[{'prompt': [{'role': 'user',
    'content': [{'type': 'image',
      'image': <PIL.Image.Image image mode=RGB size=330x110>},
     {'type': 'text',
      'text': '\nDescribe the color patch on the right so that another model can identify it among three patches.\n\nRules:\n\n- Do not use left, middle, right, center.\n- Describe only visual properties.\n- Mention hue, saturation, brightness, contrast, texture if useful.\n- Keep it under 30 words.\n'}]}],
  'patches': [[81, 50, 57], [7, 50, 54], [280, 50, 20]],
  'rl_order': [2, 1, 0],
  'judge_order': [1, 0, 2],
  'og_description': 'green'},
 {'prompt': [{'role': 'user',
    'content': [{'type': 'image',
      'image': <PIL.Image.Image image mode=RGB size=330x110>},
     {'type': 'text',
      'text': '\nDescribe the color patch on the right so that another model can identify it among three patches.\n\nRules:\n\n- Do not use left, middle, right, center.\n- Describe only visual properties.\n- Mention hue, saturation, brightness, contrast

### Grids

In [ ]:
####grids

scorer = BERTScorer(lang="en", rescale_with_baseline=True)

wandb.define_metric("reward/local_step")
wandb.define_metric("reward/*", step_metric="reward/local_step")

def build_prompt(target_position):
    return f"""
Describe the color grid {target_position} so that another model can identify it among three grids.

Rules:

- You may use positional word to reference colors in the grid, but not to reference the grid itself among the three grids.
- Describe only visual properties.
- Mention hue, saturation, brightness, contrast, texture if useful.
- Keep it under 60 words.
"""



def parse_position(text):
    text = text.lower().strip()

    if "left" in text:
        return "left"

    if "middle" in text or "center" in text:
        return "middle"

    if "right" in text:
        return "right"

    return "invalid"



@weave.op()
def judge_grid_from_description(image, description, judge_order):

    judge_prompt = f"""
You see an image with three color grids: left, middle, right.

A description says:
"{description}"

Which grid is described?

Answer only:
left, middle, or right.
"""

    # Hier dein Judge-VLM aufrufen
    #print(f"Judge Order in judge_grid_method: {judge_order}")
    img = build_grid_image(image, judge_order, **img_kwargs)
    #display(img)
    #display(img)
    #print(f"Generated Description: {description}")
    answer = run_vlm_judge(img, judge_prompt)
    #print(f"Judge Answer: {answer}")
    #print("-" * 50)

    return  {
        "position": parse_position(answer),
        "image": img,
    }


# def run_vlm_judge(image, prompt):
#     # Hier rufst du dein Judge-VLM mit dem Bild und Prompt auf und bekommst die Antwort zurück
#     # Zum Beispiel:
#     # answer = judge_vlm.generate(image=image, prompt=prompt, **generate_kwargs)
#     # return answer
#     response = judge_model.generate(prompt, image, **generate_kwargs_judge)
#     # Dummy-Implementierung für das Beispiel
#     return response


def run_vlm_judge(image, prompt):
    # Hier rufst du dein Judge-VLM mit dem Bild und Prompt auf und bekommst die Antwort zurück
    # Zum Beispiel:
    # answer = judge_vlm.generate(image=image, prompt=prompt, **generate_kwargs)
    # return answer
    #response = judge_model.generate(prompt, image, **generate_kwargs_judge)
    
    buffer = io.BytesIO()
    image.save(buffer, format="PNG")
    img_base64 = base64.b64encode(buffer.getvalue()).decode("utf-8")

    image_url = f"data:image/png;base64,{img_base64}"

    response = judge_model.chat.completions.create(
        model="Qwen/Qwen3.8-27B-FP8",
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": image_url
                        }
                    },
                    {
                        "type": "text",
                        "text": prompt
                    },
                ],
            }
        ],
        temperature=0.7,
        #max_tokens=800,
        extra_body={
            "chat_template_kwargs": {
                "enable_thinking": False
            },
            "thinking_token_budget": 40
        }
    )

    response = response.choices[0].message.content


    
    
    
    
    
    # Dummy-Implementierung für das Beispiel
    return response





def contains_position_words(text):

    forbidden = [
        "left",
        "middle",
        "right",
        "center",
        "centre",
    ]

    return any(w in text.lower() for w in forbidden)






@weave.op()
def compute_reward(description, image, target_position, judge_order):

    reward = 0.0

    judge_prediction = judge_grid_from_description(
        image=image,
        description=description,
        judge_order=judge_order
    )

    # Judge richtig?
    if judge_prediction == target_position:
        reward += 1.0
    else:
        reward -= 0.2

    # Cheating bestrafen
    #if contains_position_words(description):
    #    reward -= 0.5

    # Zu kurz
    word_count = len(description.split())

    # Zu lang
    #reward -= word_count * 0.015

    return {
        "reward": reward,
        "judge_prediction": judge_prediction,
        "word_count": word_count,
        "contains_position_words":
            contains_position_words(description)
    }






@weave.op()
def compute_reward_bert(description, image, target_position, judge_order, og_description):
    global scorer

    reward = 0.0

    F1, _, _ = scorer.score([description], [og_description])

    bert_weight = 0.5  # Gewichtung für den BERT-Score
    judge_weight = 1-bert_weight

    bert_reward = bert_weight * F1[0].item()
    reward += bert_reward  # F1-Score als Belohnung hinzufügen

    result = judge_grid_from_description(
        image=image,
        description=description,
        judge_order=judge_order
    )

    judge_prediction = result["position"]
    img = result["image"]

    


    # Judge richtig?
    prediction_true = judge_prediction == target_position

    if prediction_true:
        judge_reward = 1.0* judge_weight
    else:
        judge_reward = -0.2* judge_weight
    reward += judge_reward

    # Cheating bestrafen
    #if contains_position_words(description):
    #    reward -= 0.5

    # Zu kurz
    word_count = len(description.split())

    # Zu lang
    #reward -= word_count * 0.015

    return {
        "reward": reward,
        "judge_prediction": judge_prediction,
        "word_count": word_count,
        "contains_position_words":
            contains_position_words(description),
        "image": img,
        "bert_reward": bert_reward,
        "judge_reward": judge_reward,
        "prediction_true": prediction_true
    }




all_rewards = []
sample_rewards_avg = []
judge_correct_avg = []
word_count_avg = []
contains_position_words_avg = []
reward_local_step = 0




def reward_func(completions, grids, rl_order, judge_order, og_description, target, **kwargs):
    global all_rewards
    global reward_local_step

    rewards = []
    #print(type(completions[0]))
    #print(completions[0])
    for desc, img, rl_ord, judge_ord, og_desc, tgt in zip(
        completions,
        grids,
        rl_order,
        judge_order,
        og_description,
        target
    ):
        desc = desc[0]["content"]
        target_position = None
        if judge_ord[0] == tgt:
            target_position = "left"
        elif judge_ord[1] == tgt:
            target_position = "middle"
        elif judge_ord[2] == tgt:
            target_position = "right"



        """result = compute_reward(
            description=desc,
            image=img,
            target_position=target_position,
            judge_order=judge_ord
        )"""

        result = compute_reward_bert(
            description=desc,
            image=img,
            target_position=target_position,
            judge_order=judge_ord,
            og_description=og_desc
        )

        #display(result["image"])
        #print(f"OG Description: {og_desc}")
        #print(f"Description: {desc}")
        #print(f"Target: {target_position}")
        #print(f"RL Order: {rl_ord}")
        #print(f"Judge Order: {judge_ord}")
        #print(f"Judge Prediction: {result['judge_prediction']}")
        
        #print("___________________________________________________________________")

        rewards.append(result["reward"])

        reward_local_step += 1

        wandb.log({
            "reward/local_step": 
                reward_local_step,
                
            "reward/sample_reward":
                result["reward"],

            "reward/judge_correct":
                int(result["judge_prediction"] == target_position),

            "reward/word_count":
                result["word_count"],

            "reward/contains_position_words":
                int(result["contains_position_words"])
        })
    all_rewards.append(rewards)
    return rewards


In [ ]:
##grids

train_dataset = []
for i, entry in dataset_train.iterrows():
    #print(f"Entry {i}: {entry}")



    target = entry.target
    if entry.speaker_order[0] == target:
        target_position = "on the left"
    elif entry.speaker_order[1] == target:
        target_position = "in the middle"
    elif entry.speaker_order[2] == target:
        target_position = "on the right"
    img = build_grid_image(entry.objs, entry.speaker_order, **img_kwargs)
    prompt = build_prompt(target_position)

    utterances = entry.utterances
    conversation = ""
    for utt in utterances:
        if conversation == "":
            conversation += f"{utt[1]}"
        else:
            conversation += f", {utt[1]}"
    conversation = re.sub(r"[^A-Za-zÄÖÜäöüß,\s]", "", conversation)
    

    
    #print(f"Prompt: {prompt}")
    #display(img)
    sample = {
        "prompt": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "image", 
                        "image": img,
                    },
                    {
                        "type": "text",
                        "text": prompt
                    }
                ]
            }
        ],

        "grids": entry.objs,
        "rl_order": entry.speaker_order,
        "judge_order": entry.listener_order,
        "og_description": conversation,
        "target": target
    }
    train_dataset.append(sample)

train_dataset

[{'prompt': [{'role': 'user',
    'content': [{'type': 'image',
      'image': <PIL.Image.Image image mode=RGB size=1110x370>},
     {'type': 'text',
      'text': '\nDescribe the color grid in the middle so that another model can identify it among three grids.\n\nRules:\n\n- You may use positional word to reference colors in the grid, but not to reference the grid itself among the three grids.\n- Describe only visual properties.\n- Mention hue, saturation, brightness, contrast, texture if useful.\n- Keep it under 60 words.\n'}]}],
  'grids': [{'shapes': [{'color': [284, 23, 50]},
     {'color': [79, 20, 50]},
     {'color': [46, 44, 50]},
     {'color': [317, 3, 50]},
     {'color': [344, 27, 50]},
     {'color': [187, 28, 50]},
     {'color': [72, 71, 50]},
     {'color': [156, 93, 50]},
     {'color': [70, 77, 50]}],
    'gridDimension': 3,
    'cellLength': 40},
   {'shapes': [{'color': [212, 26, 50]},
     {'color': [271, 95, 50]},
     {'color': [237, 8, 50]},
     {'color': [82,

# 5. Training

In [13]:


training_args = GRPOConfig(

    output_dir="./qwen2vl_patch_grpo",

    report_to="wandb",

    logging_steps=1,

    learning_rate=1e-6,

    per_device_train_batch_size=1,

    gradient_accumulation_steps=8,

    num_generations=8,

    max_completion_length=80,

    num_train_epochs=1,

    bf16=True,

    save_steps=200,
)

In [ ]:
class TimedGRPOTrainer(GRPOTrainer):

    def _generate_and_score_completions(self, inputs):

        t0 = time.perf_counter()

        result = super()._generate_and_score_completions(inputs)

        t1 = time.perf_counter()

        print(
            f"Generate + Reward + Scoring: "
            f"{t1 - t0:.2f}s"
        )

        return result

trainer = TimedGRPOTrainer(
    model=hf_model,
    args=training_args,
    train_dataset=train_dataset,
    reward_funcs=reward_func,
    processing_class=processor,
)

trainer.train()
wandb.finish()
weave.finish

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None, 'pad_token_id': 151643}.
/home/alesau/miniconda3/envs/mllm/lib/python3.14/site-packages/bert_score/scorer.py:139: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  self._baseline_vals = torch.from_numpy(
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a042d8-f3bc-7299-b31e-4a6a70c28966
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01

Reward total: 38.28s
Generate + Reward + Scoring: 47.07s


Step,Training Loss
1,0.636684
2,0.167906
3,0.396891
4,0.350886
5,-0.454975
6,0.315907
7,0.530593


weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a042d9-a440-72c9-a5ea-0cad6d846ec3
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a042d9-c6a1-7ee8-a9aa-7385cb7ac3ce
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a042d9-d653-77d2-9d45-b0411ac7049f
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a042d9-f503-76e8-bf58-a00cd0809c73
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a042d9-fede-7734-8ac6-04a9acc216a1
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a042da-0d1d-7a8d-96c9-d9f9f266750d
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a042da-2432-737f-a3e4-2293b73a437f
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a042da-3697-7255-aa0f-9f46bf5633c5


Reward total: 46.91s
Generate + Reward + Scoring: 49.20s


...

weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a042dd-d243-76e3-be30-a58b31d8bed8
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a042dd-d378-7eec-a871-82d76f75ab37
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a042dd-e7b1-7841-ac83-6ef0c4dd528a
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a042de-07dd-7701-acf3-8ba68a0501d3
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a042de-10f9-76db-a6a4-9e51bfad5505
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a042de-1965-7e1b-8bab-0278ff9d5cf5
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a042de-395b-72e5-b209-f4b384cf2bc7
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a042de-4239-72d1-bc7b-8ee96932067f


Reward total: 34.43s
Generate + Reward + Scoring: 40.36s


weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a042de-7b47-7f07-ba10-de7e43c46063
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a042de-7c85-7b97-afa2-4e2ba563a445
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a042de-9011-7c78-bbfc-6028fbde6c0c
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a042de-9139-725b-a8ef-cd1ef99f1910
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a042de-b155-76c2-9d02-6efa9154ca14
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a042de-b28e-7434-b02f-99d27fb66318
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a042de-bb16-7d17-a9c0-0605cfd51b3b
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a042de-c253-7a6d-a745-4fe0da50885e


Reward total: 18.94s
Generate + Reward + Scoring: 22.96s


weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a042de-e7a8-7cdd-a9db-8a96c08119a5
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a042de-ec04-7b2d-8ef5-e93ccd279cb2
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a042de-fdd6-7957-a74c-168ae8015778
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a042df-0736-7ade-a5f2-da84d02c4ce3
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a042df-27a6-7658-a83f-e7f1f831127d
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a042df-4460-7111-9694-cb8fde9b161e
weave: 🍩 https://wandb.ai/alex-esau98/mllm/r/call/01a042df-6252-716d-942f-7c109108511c


In [10]:
#GRID
#3.5-4B JUDGE   10:47:56

#Patches
#3-5-9B JUDGE   ca. 40h

trainer = GRPOTrainer(
    model=hf_model,
    args=training_args,
    train_dataset=train_dataset,
    reward_funcs=reward_func,
    processing_class=processor,
)

trainer.train()

wandb.finish()
weave.finish()

NameError: name 'training_args' is not defined

In [15]:
wandb.finish()
weave.finish()

Error in callback <bound method _WandbInit._pre_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7f152ae746e0>> (for pre_run_cell), with arguments args (<ExecutionInfo object at 7f14445199c0, raw_cell="wandb.finish()
weave.finish()" transformed_cell="wandb.finish()
weave.finish()
" store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://wsl%2Bubuntu/home/alesau/mllm/mllm/train_model.ipynb#X44sdnNjb2RlLXJlbW90ZQ%3D%3D>,),kwargs {}:


ConnectionResetError: Connection lost

ConnectionResetError: Connection lost

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7f152ae746e0>> (for post_run_cell), with arguments args (<ExecutionResult object at 7f1444519a70, execution_count=15 error_before_exec=None error_in_exec=Connection lost info=<ExecutionInfo object at 7f14445199c0, raw_cell="wandb.finish()
weave.finish()" transformed_cell="wandb.finish()
weave.finish()
" store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://wsl%2Bubuntu/home/alesau/mllm/mllm/train_model.ipynb#X44sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost

In [19]:
print("Judge devices:", {p.device for p in judge_model.model.parameters()})
print("RL devices:", {p.device for p in hf_model.parameters()})

print(
    "allocated:",
    torch.cuda.memory_allocated() / 1024**3,
    "GB"
)

print(
    "reserved:",
    torch.cuda.memory_reserved() / 1024**3,
    "GB"
)

Error in callback <bound method _WandbInit._pre_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7f39000b86e0>> (for pre_run_cell), with arguments args (<ExecutionInfo object at 7f380cfc1ed0, raw_cell="print("Judge devices:", {p.device for p in judge_m.." transformed_cell="print("Judge devices:", {p.device for p in judge_m.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://wsl%2Bubuntu/home/alesau/mllm/mllm/train_model.ipynb#X61sdnNjb2RlLXJlbW90ZQ%3D%3D>,),kwargs {}:


ConnectionResetError: Connection lost

Judge devices: {device(type='cuda', index=0)}
RL devices: {device(type='cuda', index=0)}
allocated: 13.657028198242188 GB
reserved: 14.025390625 GB
Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7f39000b86e0>> (for post_run_cell), with arguments args (<ExecutionResult object at 7f380cfc2750, execution_count=19 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 7f380cfc1ed0, raw_cell="print("Judge devices:", {p.device for p in judge_m.." transformed_cell="print("Judge devices:", {p.device for p in judge_m.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://wsl%2Bubuntu/home/alesau/mllm/mllm/train_model.ipynb#X61sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost

# Rest

In [26]:
dataset = train_dataset[12]
img_rl = build_grid_image(dataset["grids"], dataset["rl_order"], **img_kwargs)
img_judge = build_grid_image(dataset["grids"], dataset["judge_order"], **img_kwargs)
display(img_rl)
print("RL Order:", dataset["rl_order"])
display(img_judge)
print("Judge Order:", dataset["judge_order"])
print("OG Description:", dataset["og_description"])

KeyError: 'grids'

In [ ]:
#!pip install bert_score

scorer = BERTScorer(lang="en", rescale_with_baseline=True)



Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:


dataset_train.iloc[0].conversation[0][1]

for i in range(130, 140):
    utterances = dataset_train.iloc[i].conversation
    conversation = ""
    for utt in utterances:
        if conversation == "":
            conversation += f"{utt[1]}"
        else:
            conversation += f", {utt[1]}"
    conversation = re.sub(r"[^A-Za-zÄÖÜäöüß,\s]", "", conversation)
    print(conversation)
    #print("Target: ", dataset_train.iloc[i].target)

yellow
orange
blue
purple
Target is the orange square
purple
tan olive
green
gray
green


In [ ]:
scorer_raw = BERTScorer(lang="en", rescale_with_baseline=False)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
P, R, F1 = scorer.score(
    ["red", "red", "red", "red", "red", "red", "red", "red"],
    ["The red one", "pink", "red", "blue", "green", "red red", "red red red", "the horse rider falls on his back"],
)

print(F1)
print(P)
print(R)
#print()

#P, R, F1 = scorer_raw.score(
#    ["red", "red", "red", "red", "red", "red", "red"],
#    ["The red one", "pink", "red", "blue", "green", "red red", "red red red"],
#)

#print(F1)
#print(F1_new)
#F1[0].item()

tensor([0.2657, 0.4357, 1.0000, 0.6542, 0.5561, 0.7249, 0.5524, 0.0383])
tensor([0.3139, 0.4975, 1.0000, 0.6536, 0.5553, 0.7738, 0.7098, 0.2696])
tensor([ 0.2160,  0.3734,  1.0000,  0.6536,  0.5554,  0.6759,  0.4022, -0.1761])


In [ ]:
!pip install sentence_transformers


model = SentenceTransformer("all-mpnet-base-v2")

cands = ["red"] * 7
refs = ["The red one", "pink", "red", "blue", "green", "red red", "red red red"]

emb_c = model.encode(cands, convert_to_tensor=True)
emb_r = model.encode(refs, convert_to_tensor=True)

scores = util.cos_sim(emb_c, emb_r).diagonal()
print(scores)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tensor([0.5746, 0.7402, 1.0000, 0.7394, 0.7460, 0.7848, 0.7123],
       device='cuda:0')


-uni VPN einrichten
-qwen evaluation (vs. Claude vs. Gemini vs. gemma4)

